# Check Grokking

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import sys

sys.path.append("..")
from src.utils import load_json

data1 = load_json(f"../result/random_label/cifar10_0.0.json")
data2 = load_json(f"../result/random_label/cifar10_0.25.json")
data3 = load_json(f"../result/random_label/cifar10_0.5.json")
data4 = load_json(f"../result/random_label/cifar10_0.75.json")
data5 = load_json(f"../result/random_label/cifar10_0.9.json")
data6 = load_json(f"../result/random_label/cifar10_1.0.json")

bunch = [data1, data2, data3, data4, data5, data6]

In [ ]:
# Train and Test Accuracy
train_accuracy, test_accuracy, generalization_error = [], [], []
for data in bunch:
    tmp1 = np.array(data['train_accuracy'])# [50:200]
    tmp2 = np.array(data['test_accuracy'])# [50:200]
    train_accuracy.append(tmp1)
    test_accuracy.append(tmp2)
    generalization_error.append(tmp1 - tmp2)

# Train and Test Target Linearity
train_target_linearity, test_target_linearity = [], []
for data in bunch:
    tmp = np.clip(np.array(data['train_target_linearity']).T, min=0.0)
    train_target_linearity.append(tmp) # [:,50:200])


In [ ]:
n_layers = train_target_linearity[1].shape[0]
fig, axes = plt.subplots(ncols=3, nrows=2, figsize=(10,7), sharey=True)
cmap = cm.viridis
colors = [cmap(1 - (i / (n_layers - 1))) for i in range(n_layers)]

titles = [['$p=0.0$', '$p=0.25$', '$p=0.5$'], ['$p=0.75$', '$p=0.9$', '$p=1.0$']]

for i, axe in enumerate(axes):
    for j, ax in enumerate(axe):
        # --- Create Twin Axis for Accuracy ---
        ax_acc = ax.twinx()
        ax_acc.set_ylim(0.05, 0.95)
        
        # --- Plot Target Linearity ---
        for layer_idx in range(n_layers):
            ax.plot(train_target_linearity[i * 3 + j][layer_idx], 
                    color=colors[layer_idx], 
                    linewidth=4,
                    label=f'Layer {layer_idx+1}')
        ax.set_ylim(0.0, 0.5)
            
        # Plot Accuracy as a dashed background reference
        line_train = ax_acc.plot(train_accuracy[i * 3 + j], color='blue', linestyle='--', alpha=0.4, label='Train Acc', linewidth=4.0)
        line_test = ax_acc.plot(test_accuracy[i * 3 + j], color='red', linestyle='--', alpha=0.4, label='Test Acc', linewidth=4.0)

        # Formatting
        ax.set_title(titles[i][j], weight='bold', size=16)
        ax.grid(True, which='both', linestyle=':', alpha=0.5)
        ax.tick_params(labelleft=True)
        
        if i == 0 and j == len(axe) - 1:
            # Create a combined legend
            lines, labels = ax.get_legend_handles_labels()
            lines2, labels2 = ax_acc.get_legend_handles_labels()
            ax.legend(lines + lines2, labels + labels2, loc='lower right', fontsize='small')
        
        
fig.supylabel('Target Linearity', weight='bold', size=20, x=0.02)
fig.text(1.0, 0.4, "Accuracy", weight='bold', size=20, 
         va='center', rotation='vertical', rotation_mode='anchor')
fig.suptitle('Random Label Memorization', weight='bold', size=22)
fig.supxlabel('Epochs', weight='bold', size=20)
plt.tight_layout()
plt.show()